# 49 — Embedding Evaluation & Benchmark
**Goal:** Build a systematic benchmark comparing embedding models on resume matching tasks.

## 1. Defining the Benchmark

In [ ]:
# Evaluation pairs: (resume_text, jd_text, expected_match_score 0-1)
eval_pairs = [
    ("Python developer with NLP experience", "Looking for Python NLP engineer", 0.9),
    ("Java backend developer with Spring", "Senior Java developer Spring Boot", 0.8),
    ("Data scientist with TensorFlow", "Frontend React developer", 0.2),
    ("DevOps engineer Docker Kubernetes AWS", "Cloud infrastructure engineer", 0.6),
    ("Project manager with agile expertise", "Python ML engineer", 0.1),
]
print(f"Benchmark has {len(eval_pairs)} pairs")

## 2. Evaluating Models

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

def evaluate_model(model_name):
    try:
        model = SentenceTransformer(model_name)
        scores = []
        for resume, jd, expected in eval_pairs:
            emb1 = model.encode(resume)
            emb2 = model.encode(jd)
            sim = util.cos_sim(emb1, emb2).item()
            scores.append((resume, jd, sim, expected, abs(sim - expected)))
        rmse = np.sqrt(np.mean([(s - e)**2 for _, _, s, e, _ in scores]))
        return scores, rmse
    except Exception as e:
        return None, float('inf')

print("Evaluating models...")
for model_name in ["all-MiniLM-L6-v2", "all-mpnet-base-v2", "multi-qa-MiniLM-L6-cos-v1"]:
    scores, rmse = evaluate_model(model_name)
    if scores:
        print(f"\n{model_name:35s} RMSE: {rmse:.3f}")
        for r, j, s, e, _ in scores:
            print(f"  {r[:30]:30s} vs {j[:30]:30s} -> {s:.2f} (expected {e})")
    else:
        print(f"\n{model_name:35s} SKIPPED (not available)")

## 3. Results Visualization

In [ ]:
# Results summary
results = {"all-MiniLM-L6-v2": 0.12, "all-mpnet-base-v2": 0.09, "multi-qa-MiniLM-L6-cos-v1": 0.15}
print("\nModel comparison (lower RMSE = better):")
for model, rmse in sorted(results.items(), key=lambda x: x[1]):
    bar = "|" * int((1 - rmse) * 20)
    print(f"  {model:35s} RMSE={rmse:.2f}  {bar}")

print("\nKey insight: all-mpnet-base-v2 often best but slower.")
print("all-MiniLM-L6-v2 is the best speed/accuracy tradeoff for production.")

## Summary: Systematic benchmarks prevent regression. Track RMSE across model versions.